# MIC Creep Prediction — Data Exploration
## Vivli AMR Challenge 2026

This notebook explores the BVBRC genome AMR datasets for K. pneumoniae and A. baumannii,
assessing data quality, temporal coverage, and suitability for MIC creep modeling.

## 1. Data Loading and Initial Inspection

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Define paths
project_root = Path('../')
data_raw = project_root / 'data' / 'raw'

print(f"Project root: {project_root}")
print(f"Data directory: {data_raw}")
print(f"Files available:")
for f in sorted(data_raw.glob('*')):
    print(f"  - {f.name} ({f.stat().st_size / (1024**2):.1f} MB)")

Project root: ..
Data directory: ../data/raw
Files available:
  - Acinetobacter_baumannii_BVBRC_genome_amr.csv (90.6 MB)
  - Acinetobacter_baumannii_isolates.tsv (24.3 MB)
  - Klebsiella_pneumoniae_BVBRC_genome_amr.csv (359.4 MB)
  - Klebsiella_pneumoniae_isolates.tsv (96.5 MB)


## 2. Loading BVBRC Datasets

**Key Notes:**
- Data is from BVBRC (Bacterial & Viral Bioinformatics Resource Center)
- Contains genome-level AMR annotations, not clinical isolate MIC measurements
- We need to extract Meropenem phenotype/MIC data from `*_BVBRC_genome_amr.csv`
- Isolate metadata (dates, location) comes from `*_isolates.tsv`

In [22]:
print("Loading Klebsiella pneumoniae AMR data...")
kp_amr = pd.read_csv(data_raw / 'Klebsiella_pneumoniae_BVBRC_genome_amr.csv', low_memory=False)
print(f"Shape: {kp_amr.shape}")

print("\nLoading Klebsiella pneumoniae isolate metadata...")
kp_isolates = pd.read_csv(data_raw / 'Klebsiella_pneumoniae_isolates.tsv', sep='\t', low_memory=False)
print(f"Shape: {kp_isolates.shape}")

print("\nLoading Acinetobacter baumannii AMR data...")
ab_amr = pd.read_csv(data_raw / 'Acinetobacter_baumannii_BVBRC_genome_amr.csv', low_memory=False)
print(f"Shape: {ab_amr.shape}")

print("\nLoading Acinetobacter baumannii isolate metadata...")
ab_isolates = pd.read_csv(data_raw / 'Acinetobacter_baumannii_isolates.tsv', sep='\t', low_memory=False)
print(f"Shape: {ab_isolates.shape}")

Loading Klebsiella pneumoniae AMR data...
Shape: (2006311, 21)

Loading Klebsiella pneumoniae isolate metadata...
Shape: (290722, 18)

Loading Acinetobacter baumannii AMR data...
Shape: (515916, 21)

Loading Acinetobacter baumannii isolate metadata...
Shape: (55910, 18)


## 3. Exploring Dataset Structure

### 3.1 Klebsiella AMR Dataset

In [24]:
print("=" * 80)
print("KLEBSIELLA PNEUMONIAE — AMR Dataset")
print("=" * 80)
print(f"\nColumns: {kp_amr.shape[1]}")
print(kp_amr.columns.tolist())
print(f"\nData types:\n{kp_amr.dtypes}")
print(f"\nFirst 3 rows:")
print(kp_amr.head(3).to_string())
print(f"\nUnique Antibiotics: {kp_amr['Antibiotic'].nunique()}")
print(f"\nAntibiotics available:")
print(kp_amr['Antibiotic'].value_counts().head(15))

KLEBSIELLA PNEUMONIAE — AMR Dataset

Columns: 21
['Taxon ID', 'Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'Measurement', 'Measurement Sign', 'Measurement Value', 'Measurement Unit', 'Laboratory Typing Method', 'Laboratory Typing Method Version', 'Laboratory Typing Platform', 'Vendor', 'Testing Standard', 'Testing Standard Year', 'Computational Method', 'Computational Method Version', 'Computational Method Performance', 'Evidence', 'Source', 'PubMed']

Data types:
Taxon ID                              int64
Genome ID                           float64
Genome Name                             str
Antibiotic                              str
Resistant Phenotype                     str
Measurement                             str
Measurement Sign                        str
Measurement Value                       str
Measurement Unit                        str
Laboratory Typing Method                str
Laboratory Typing Method Version        str
Laboratory Typing Platform 

In [25]:
print("\n" + "=" * 80)
print("MEROPENEM-SPECIFIC ANALYSIS")
print("=" * 80)
kp_mero = kp_amr[kp_amr['Antibiotic'] == 'Meropenem'].copy()
print(f"\nKlebsiella + Meropenem records: {len(kp_mero)} ({100*len(kp_mero)/len(kp_amr):.1f}%)")

print(f"\nMeasurement columns:")
print(f"  - Measurement Sign values: {kp_mero['Measurement Sign'].unique()}")
print(f"  - Measurement Unit: {kp_mero['Measurement Unit'].unique()}")
print(f"  - Measurement Value range: {kp_mero['Measurement Value'].min()} to {kp_mero['Measurement Value'].max()}")
print(f"  - Resistant Phenotype: {kp_mero['Resistant Phenotype'].unique()}")

print(f"\nMeasurement Value distribution for Meropenem:")
print(kp_mero['Measurement Value'].describe())


MEROPENEM-SPECIFIC ANALYSIS

Klebsiella + Meropenem records: 0 (0.0%)

Measurement columns:
  - Measurement Sign values: <StringArray>
[]
Length: 0, dtype: str
  - Measurement Unit: <StringArray>
[]
Length: 0, dtype: str
  - Measurement Value range: nan to nan
  - Resistant Phenotype: <StringArray>
[]
Length: 0, dtype: str

Measurement Value distribution for Meropenem:
count       0
unique      0
top       NaN
freq      NaN
Name: Measurement Value, dtype: object


In [26]:
print("\n" + "=" * 80)
print("ISOLATE METADATA — Klebsiella")
print("=" * 80)
print(f"Columns: {kp_isolates.shape[1]}")
print(kp_isolates.columns.tolist())
print(f"\nFirst row (non-null values):")
print(kp_isolates.iloc[0])
print(f"\nMissing values (%):")
print((kp_isolates.isnull().sum() / len(kp_isolates) * 100).sort_values(ascending=False).head(10))


ISOLATE METADATA — Klebsiella
Columns: 20
['#Organism group', 'AST phenotypes', 'Strain', 'Isolate identifiers', 'Serovar', 'Isolate', 'Create date', 'Location', 'Isolation source', 'Isolation type', 'Food origin', 'SNP cluster', 'Min-same', 'Min-diff', 'BioSample', 'Assembly', 'AMR genotypes', 'Computed types', 'IsolateCreatedate', 'year']

First row (non-null values):
#Organism group                                      Salmonella enterica
AST phenotypes                                                       NaN
Strain                                                           JNQH953
Isolate identifiers                                              JNQH953
Serovar                                                      Enteritidis
Isolate                                                   PDT001913736.1
Create date                                         2023-10-03T18:01:41Z
Location                                                  China:Shandong
Isolation source                     bronc

In [27]:
print("\n" + "=" * 80)
print("TEMPORAL COVERAGE")
print("=" * 80)

# Parse 'Create date' from isolates
kp_isolates['IsolateCreatedate'] = pd.to_datetime(kp_isolates['Create date'], errors='coerce')
kp_isolates['year'] = kp_isolates['IsolateCreatedate'].dt.year

print(f"\nKlebsiella isolates with valid dates: {kp_isolates['IsolateCreatedate'].notna().sum()}/{len(kp_isolates)}")
print(f"Year range: {kp_isolates['year'].min()} to {kp_isolates['year'].max()}")
print(f"\nRecords per year:")
print(kp_isolates['year'].value_counts().sort_index().tail(15))


TEMPORAL COVERAGE

Klebsiella isolates with valid dates: 290722/290722
Year range: 2010 to 2026

Records per year:
year
2012      130
2013      154
2014      268
2015     4794
2016     3368
2017     2495
2018     5135
2019    26115
2020    14393
2021    27392
2022    23898
2023    49979
2024    42165
2025    70298
2026    20081
Name: count, dtype: int64


## 4. Key Findings & Next Steps

In [28]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 Data Overview:")
print(f"  • K. pneumoniae AMR records: {len(kp_amr):,} (Meropenem: {len(kp_mero):,})")
print(f"  • K. pneumoniae isolates: {len(kp_isolates):,}")
print(f"  • A. baumannii AMR records: {len(ab_amr):,}")
print(f"  • A. baumannii isolates: {len(ab_isolates):,}")

print("\n⏰ Temporal Coverage:")
if 'year' not in kp_isolates.columns:
    kp_isolates['IsolateCreatedate'] = pd.to_datetime(kp_isolates['Create date'], errors='coerce')
    kp_isolates['year'] = kp_isolates['IsolateCreatedate'].dt.year

print(f"  • K. pneumoniae: {kp_isolates['year'].min()} to {kp_isolates['year'].max()}")
valid_kp = kp_isolates['IsolateCreatedate'].notna().sum()
print(f"  • Date coverage: {valid_kp}/{len(kp_isolates)} records ({100*valid_kp/len(kp_isolates):.1f}%)")

print("\n🧬 Meropenem Data:")
print(f"  • Records with MIC measurements: {(kp_mero['Measurement Value'].notna()).sum():,}")
print(f"  • MIC value range: {kp_mero['Measurement Value'].min()} to {kp_mero['Measurement Value'].max()} mg/L")
print(f"  • Phenotypes: {', '.join(kp_mero['Resistant Phenotype'].dropna().unique())}")

print("\n⚠️  IMPORTANT LIMITATIONS:")
print("  • BVBRC data = genome-level AMR annotations (genotypic), not clinical MIC measurements")
print("  • No temporal metadata linking AMR records to isolate collection dates")
print("  • This is proxy data; awaiting actual ATLAS/SENTRY datasets with clinical MIC values")
print("  • Cannot perform time-aware training/test split without joining on Genome ID")

ANALYSIS SUMMARY

📊 Data Overview:
  • K. pneumoniae AMR records: 2,006,311 (Meropenem: 0)
  • K. pneumoniae isolates: 290,722
  • A. baumannii AMR records: 515,916
  • A. baumannii isolates: 55,910

⏰ Temporal Coverage:
  • K. pneumoniae: 2010 to 2026
  • Date coverage: 290722/290722 records (100.0%)

🧬 Meropenem Data:
  • Records with MIC measurements: 0
  • MIC value range: nan to nan mg/L
  • Phenotypes: 

⚠️  IMPORTANT LIMITATIONS:
  • BVBRC data = genome-level AMR annotations (genotypic), not clinical MIC measurements
  • No temporal metadata linking AMR records to isolate collection dates
  • This is proxy data; awaiting actual ATLAS/SENTRY datasets with clinical MIC values
  • Cannot perform time-aware training/test split without joining on Genome ID


## 5. Recommended Next Steps

### Immediate (This Week)   
1. **Create data pipeline modules** (`src/data/`)
   - `loader.py`: Load BVBRC & isolates, join on Genome ID
   - `preprocessor.py`: Handle censored MIC values (">8" → 16, "<=0.5" → 0.25)
   - Extract year from dates, parse location to country

### Short-term (Awaiting Real Data)
3. **Build with BVBRC data as proof-of-concept**
   - Join isolates metadata with AMR records
   - Filter for K. pneumoniae + Meropenem
   - Create year/country aggregations
   - Plot MIC₉₀ trends by year

4. **EDA & Feature Engineering**
   - Visualize MIC distribution by year (should show upward trend if creep exists)
   - Extract country codes from Location field
   - Create age_group, specimen_type features (source limitations apply to genomic data)
   - Log₂-transform MIC values

### Medium-term (When ATLAS/SENTRY data arrives)
5. **Re-run pipeline on real data**
   - Update `loader.py` for actual data schema
   - Apply time-aware train/test split (2004–2018 vs 2019–2022)
   - Train XGBoost + Random Forest
   - Compute RMSE, MAE, MIC₉₀ trends

6. **Model Explainability**
   - Extract SHAP values
   - Validate with domain expert for biological plausibility

7. **Deployment**
   - Package model, push to Hugging Face Hub
   - Build FastAPI backend
   - Deploy Next.js frontend